# Infraestructuras Computacionales para el Procesamiento de datos masivos

**Ejercicio 1, Modulo 2 Tema 2**


Este cuaderno está diseñado desde un enfoque académico diseñado para poner en práctica los conceptos explicados en el Tema 2: Programación de Apliacaciones Spark del Modulo 2 de la asignatura.


## Objetivos generales del cuaderno

A lo largo de este notebook se persiguen los siguientes objetivos:

1. Cargar y comprender la estructura real del dataset.
2. Aplicar procesos de preparación y tipado de datos en Spark.
3. Enriquecer la tabla principal con dimensiones auxiliares.
4. Construir análisis agregados de interés para negocio.
5. Aplicar técnicas avanzadas de Spark:
   - `groupBy`
   - `pivot`
   - funciones de ventana
   - `broadcast join`
   - `left_semi` y `left_anti`
   - joins por rango temporal
6. Relacionar los resultados obtenidos con un contexto real de analítica de ventas.

## Dataset utilizado

Este notebook está adaptado a los tres ficheros reales del dataset de Kaggle:

- `sales.csv`
- `product_hierarchy.csv`
- `store_cities.csv`

## Columnas principales

### `sales.csv`
- `store_id`
- `product_id`
- `date`
- `sales`
- `revenue`
- `stock`
- `price`
- `promo_type_1`
- `promo_bin_1`
- `promo_type_2`
- `promo_bin_2`
- `promo_discount_2`
- `promo_discount_type_2`

### `product_hierarchy.csv`
- `product_id`
- `product_length`
- `product_depth`
- `product_width`
- `hierarchy1_id`
- `hierarchy2_id`
- `hierarchy3_id`
- `hierarchy4_id`
- `hierarchy5_id`

### `store_cities.csv`
- `store_id`
- `storetype_id`
- `store_size`
- `city_id`

### Variable adicional relevante
- `train_or_test`: etiqueta de partición para entrenamiento y prueba.

## Recomendación de uso

Se recomienda ejecutar el cuaderno **bloque a bloque**, revisando siempre:
- el esquema resultante,
- las primeras filas,
- y la interpretación analítica de cada salida.



## 1. Creación de la sesión de Spark

En cualquier aplicación PySpark, el punto de entrada es la `SparkSession`.  
Esta sesión representa la conexión lógica con el motor de Spark y permite leer datos, transformarlos y ejecutar acciones.

En un contexto académico, este paso es importante porque muestra el inicio del pipeline analítico y el arranque del entorno distribuido.




In [ ]:
!pip install pyspark==3.5.1

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Retail_Sales_Analysis")
    .getOrCreate()
)

spark

### Ejercicio propuesto

1. Explica con tus palabras qué papel desempeña `SparkSession` dentro de una aplicación PySpark.
2. Indica qué diferencias conceptuales existen entre `SparkSession` y `SparkContext`.
3. Comprueba en tu entorno qué versión de Spark estás ejecutando.


## 2. Definición de rutas de entrada

En este bloque se definen las rutas de los tres ficheros del dataset.  



In [ ]:
sales_path = "DatasetTema2/sales.csv"
product_path = "DatasetTema2/product_hierarchy.csv"
stores_path = "DatasetTema2/store_cities.csv"

### Ejercicio propuesto

1. Modifica las rutas para adaptarlas a tu entorno local o al entorno Docker de Spark.
2. Explica por qué es recomendable no “hardcodear” rutas complejas en múltiples celdas del notebook.


## 3. Carga de los ficheros CSV

En este bloque se cargan los tres datasets.  
Se utiliza `header=True` para interpretar correctamente la cabecera y `inferSchema=True` para que Spark intente deducir los tipos de datos automáticamente.

Aunque `inferSchema` es muy útil para exploración, en proyectos más robustos suele ser recomendable definir esquemas explícitos para mejorar:
- la reproducibilidad,
- el rendimiento de lectura,
- y el control sobre los tipos.


In [ ]:
sales_df = spark.read.csv(sales_path, header=True, inferSchema=True)
product_df = spark.read.csv(product_path, header=True, inferSchema=True)
stores_df = spark.read.csv(stores_path, header=True, inferSchema=True)

print("Esquema de sales.csv")
sales_df.printSchema()

print("Esquema de product_hierarchy.csv")
product_df.printSchema()

print("Esquema de store_cities.csv")
stores_df.printSchema()

### Ejercicio propuesto

1. Revisa los esquemas obtenidos y anota qué columnas Spark ha interpretado como numéricas y cuáles como texto.
2. Indica qué riesgos puede tener depender únicamente de `inferSchema=True`.
3. Elabora una breve propuesta de esquema explícito para `sales.csv`.


## 4. Exploración inicial de los datos

Antes de transformar un dataset es fundamental inspeccionar algunas filas.  
Esta práctica permite:
- detectar valores nulos,
- entender la granularidad,
- validar nombres de columnas,
- y confirmar si la estructura coincide con la documentación del dataset.


In [ ]:
sales_df.show(5, truncate=False)
product_df.show(5, truncate=False)
stores_df.show(5, truncate=False)

### Ejercicio propuesto

1. Describe qué representa una fila de `sales.csv`.
2. Indica si la granularidad del dataset parece ser diaria, semanal o transaccional.
3. ¿Qué columnas del dataset consideras hechos y cuáles dimensiones?


## 5. Tipado y preparación de `sales.csv`

Este bloque transforma `sales_df` para preparar los datos antes del análisis.

### Transformaciones realizadas

- Conversión de `date` al tipo `date`.
- Conversión explícita de variables numéricas.
- Creación de columnas temporales (`year`, `month`, `weekofyear`).
- Generación de indicadores auxiliares:
  - `is_stockout`: marca si el stock final del día es cero o negativo.
  - `avg_ticket_est`: estimación del ingreso medio por unidad vendida.

Este tipo de preparación es una etapa clásica de ingeniería de datos y análisis exploratorio.


In [ ]:
sales_df = (
    sales_df
    .withColumn("date", F.to_date("date"))
    .withColumn("sales", F.col("sales").cast("double"))
    .withColumn("revenue", F.col("revenue").cast("double"))
    .withColumn("stock", F.col("stock").cast("double"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("promo_discount_2", F.col("promo_discount_2").cast("double"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("weekofyear", F.weekofyear("date"))
    .withColumn("is_stockout", F.when(F.col("stock") <= 0, 1).otherwise(0))
    .withColumn("avg_ticket_est", F.when(F.col("sales") != 0, F.col("revenue") / F.col("sales")))
)

sales_df.printSchema()
sales_df.show(5, truncate=False)

### Comentario analítico

La variable `avg_ticket_est` no representa un ticket por cliente, ya que el dataset no incluye clientes individuales.  
Sin embargo, sí puede interpretarse como una **aproximación al ingreso medio por unidad vendida** en cada combinación tienda-producto-fecha.

La variable `is_stockout` será especialmente útil para analizar roturas de stock, un aspecto muy relevante en entornos retail.


### Ejercicio propuesto

1. Explica la diferencia entre convertir `date` a tipo `date` y dejarla como texto.
2. Justifica por qué se crea la columna `is_stockout`.
3. Propón dos columnas derivadas adicionales que podrían ser útiles para el análisis.


## 6. Enriquecimiento con dimensiones de producto y tienda

Una práctica habitual en analítica y Data Warehousing es trabajar con una tabla de hechos y varias dimensiones.

En este caso:
- `sales_df` actúa como tabla principal de hechos.
- `product_df` aporta jerarquía y dimensiones físicas del producto.
- `stores_df` aporta características de la tienda.

La unión se realiza con `broadcast`, una optimización especialmente útil cuando las tablas dimensionales son lo bastante pequeñas como para replicarse en memoria en los ejecutores.


In [ ]:
retail_df = (
    sales_df
    .join(F.broadcast(product_df), "product_id", "left")
    .join(F.broadcast(stores_df), "store_id", "left")
)

retail_df.printSchema()
retail_df.show(5, truncate=False)

### Comentario analítico

Gracias a este enriquecimiento, el análisis deja de estar limitado a tienda, producto y fecha, y pasa a incorporar:
- jerarquías de producto,
- tamaño de tienda,
- tipo de tienda,
- ciudad.

Esto permite formular preguntas mucho más ricas, por ejemplo:
- ¿qué jerarquías venden más?
- ¿qué tamaños de tienda presentan más roturas de stock?
- ¿hay diferencias entre ciudades?


### Ejercicio propuesto

1. Explica por qué este patrón de unión se parece a un modelo estrella.
2. ¿Qué ventaja tiene usar `broadcast` en este caso?
3. ¿Qué ocurriría si se hiciera `broadcast` sobre una tabla demasiado grande?


## 7. Comprobación de columnas tras el enriquecimiento

En datasets reales es recomendable revisar las columnas finales antes de continuar.  
Este paso ayuda a:
- validar que las uniones se han realizado correctamente,
- comprobar que no hay colisiones inesperadas,
- y confirmar que las columnas necesarias están disponibles.


In [ ]:
retail_df.columns

### Ejercicio propuesto

1. Localiza qué columnas pertenecen a ventas, cuáles a productos y cuáles a tiendas.
2. Clasifica las columnas en métricas, identificadores y atributos descriptivos.


## 8. KPI inicial: ventas y facturación por tienda

Una primera vista útil consiste en agregar por `store_id` para obtener indicadores agregados de rendimiento.

### Métricas calculadas

- `total_sales_qty`: suma de unidades vendidas.
- `total_revenue`: suma de ingresos.
- `avg_price`: precio medio.
- `avg_stock`: stock medio final.

Este tipo de tabla permite identificar tiendas con alto volumen, alto ingreso o posibles diferencias operativas.


In [ ]:
kpi_store = (
    retail_df
    .groupBy("store_id")
    .agg(
        F.sum("sales").alias("total_sales_qty"),
        F.sum("revenue").alias("total_revenue"),
        F.avg("price").alias("avg_price"),
        F.avg("stock").alias("avg_stock")
    )
    .orderBy(F.desc("total_revenue"))
)

kpi_store.show(20, truncate=False)

### Ejercicio propuesto

1. Identifica las tiendas con mayor facturación.
2. ¿Existe relación aparente entre `avg_stock` y `total_revenue`?
3. Propón una visualización adecuada para representar este resultado en el TFG.


## 9. Análisis por ciudad, tipo y tamaño de tienda

En este bloque se amplía la agregación anterior utilizando atributos de la dimensión de tienda:
- `city_id`
- `storetype_id`
- `store_size`

Esto permite estudiar el comportamiento agregado por perfil de tienda y contexto geográfico.


In [ ]:
kpi_city_storetype = (
    retail_df
    .groupBy("city_id", "storetype_id", "store_size")
    .agg(
        F.sum("sales").alias("sales_qty"),
        F.sum("revenue").alias("revenue_total"),
        F.avg("price").alias("avg_price")
    )
    .orderBy(F.desc("revenue_total"))
)

kpi_city_storetype.show(20, truncate=False)

### Ejercicio propuesto

1. Señala qué combinación de ciudad, tipo y tamaño parece más rentable.
2. ¿Qué utilidad tendría este análisis para un responsable comercial?
3. ¿Qué limitaciones tiene este resultado si `city_id` no está descrito semánticamente?


## 10. Tabla dinámica (pivot) por tipo de promoción


El objetivo es construir una tabla donde:
- las filas sean las tiendas,
- las columnas sean los tipos de promoción,
- y el valor agregado sea la suma de `revenue`.

Este patrón es útil para comparar el impacto económico de distintos tipos promocionales.


In [ ]:
promo_type_1_values = [r[0] for r in retail_df.select("promo_type_1").distinct().orderBy("promo_type_1").collect() if r[0] is not None]
promo_type_1_values[:20]

In [ ]:
df_pivot_promo1 = (
    retail_df
    .groupBy("store_id")
    .pivot("promo_type_1", promo_type_1_values)
    .agg(F.sum("revenue"))
)

df_pivot_promo1.show(20, truncate=False)

### Comentario analítico

Las tablas pivot son especialmente útiles para:
- reporting,
- dashboards,
- comparación entre grupos,
- y exportación a herramientas de visualización.

No obstante, pueden ser costosas si la cardinalidad de la columna pivotada es alta.


### Ejercicio propuesto

1. Explica qué diferencias hay entre un `groupBy` tradicional y un `pivot`.
2. Repite el análisis usando `promo_type_2`.
3. Sustituye la suma de `revenue` por la suma de `sales` y compara los resultados.


## 11. Top 3 productos por jerarquía principal

En lugar de utilizar una categoría genérica, aquí se aprovecha la jerarquía real de producto mediante `hierarchy1_id`.

El procedimiento consiste en:
1. agregar ventas por `hierarchy1_id` y `product_id`,
2. ordenar por facturación,
3. asignar un ranking con `row_number()`,
4. filtrar el Top 3 por grupo.

Este patrón es uno de los usos clásicos de las funciones de ventana.


In [ ]:
df_prod_h1 = (
    retail_df
    .groupBy("hierarchy1_id", "product_id")
    .agg(
        F.sum("revenue").alias("revenue_total"),
        F.sum("sales").alias("sales_qty")
    )
)

w_h1 = Window.partitionBy("hierarchy1_id").orderBy(F.desc("revenue_total"))

df_top_h1 = (
    df_prod_h1
    .withColumn("rn", F.row_number().over(w_h1))
    .filter(F.col("rn") <= 3)
    .drop("rn")
    .orderBy("hierarchy1_id", F.desc("revenue_total"))
)

df_top_h1.show(50, truncate=False)

### Ejercicio propuesto

1. Explica por qué se utiliza `partitionBy("hierarchy1_id")`.
2. ¿Qué cambiaría si se usara `rank()` en lugar de `row_number()`?
3. Repite el ejercicio con `hierarchy2_id`.


## 12. Top 3 productos por tienda

Este análisis cambia la perspectiva: ahora no interesa la jerarquía global, sino el rendimiento de producto dentro de cada tienda.

Este tipo de ranking puede ser útil para:
- gestión del surtido,
- optimización de inventario,
- detección de productos estrella por ubicación.


In [ ]:
df_prod_store = (
    retail_df
    .groupBy("store_id", "product_id")
    .agg(
        F.sum("revenue").alias("revenue_total"),
        F.sum("sales").alias("sales_qty")
    )
)

w_store = Window.partitionBy("store_id").orderBy(F.desc("revenue_total"))

df_top_store = (
    df_prod_store
    .withColumn("rn", F.row_number().over(w_store))
    .filter(F.col("rn") <= 3)
    .drop("rn")
)

df_top_store.show(50, truncate=False)

### Ejercicio propuesto

1. Identifica si los productos top se repiten entre tiendas o si existe heterogeneidad.
2. Propón una interpretación comercial para este resultado.
3. Añade el `store_size` al resultado final para enriquecer el análisis.


## 13. Media móvil de 7 días por tienda y producto

Dado que el dataset tiene granularidad diaria, una ventana temporal móvil es muy adecuada.

En este caso se calcula la media de `revenue` en los últimos 7 días para cada combinación de:
- `store_id`
- `product_id`

Este tipo de métrica ayuda a suavizar fluctuaciones diarias y detectar tendencias recientes.


In [ ]:
w7d = (
    Window
    .partitionBy("store_id", "product_id")
    .orderBy(F.col("date").cast("timestamp").cast("long"))
    .rangeBetween(-7 * 24 * 3600, 0)
)

df_roll_7d = retail_df.withColumn("revenue_avg_7d", F.avg("revenue").over(w7d))

df_roll_7d.select(
    "store_id", "product_id", "date", "revenue", "revenue_avg_7d"
).show(30, truncate=False)

### Comentario analítico

La media móvil permite reducir ruido y comparar el comportamiento actual con una tendencia reciente.  
Es especialmente útil en retail para:
- seguimiento de producto,
- detección de cambios de demanda,
- análisis del efecto de promociones.


### Ejercicio propuesto

1. Explica la diferencia entre `rangeBetween` y `rowsBetween`.
2. Justifica por qué en este caso se usa una ventana temporal y no una ventana por número de filas.
3. Calcula también una media móvil de 14 días.


## 14. Media móvil de 30 días

Una ventana más amplia permite observar tendencias menos sensibles al ruido diario.

Comparar la media móvil de 7 días con la de 30 días puede resultar útil para detectar:
- aceleraciones,
- desaceleraciones,
- o comportamientos anómalos.


In [ ]:
w30d = (
    Window
    .partitionBy("store_id", "product_id")
    .orderBy(F.col("date").cast("timestamp").cast("long"))
    .rangeBetween(-30 * 24 * 3600, 0)
)

df_roll_30d = retail_df.withColumn("revenue_avg_30d", F.avg("revenue").over(w30d))

df_roll_30d.select(
    "store_id", "product_id", "date", "revenue", "revenue_avg_30d"
).show(30, truncate=False)

### Ejercicio propuesto

1. Compara conceptualmente las ventanas de 7 y 30 días.
2. ¿Cuál usarías para seguimiento operativo diario y cuál para reporting mensual?
3. Calcula una columna adicional con la diferencia entre `revenue` y `revenue_avg_30d`.


## 15. Análisis de roturas de stock

Una rotura de stock implica que al final del día el producto queda sin inventario disponible.  
Esto puede ser indicio de:
- alta demanda,
- mala planificación,
- o problemas de reposición.

En este bloque se calcula:
- el número de días con rotura de stock,
- el número total de días observados,
- y el ratio de rotura por combinación tienda-producto.


In [ ]:
stockout_kpi = (
    retail_df
    .groupBy("store_id", "product_id")
    .agg(
        F.sum("is_stockout").alias("days_stockout"),
        F.count("*").alias("days_total"),
        (F.sum("is_stockout") / F.count("*")).alias("stockout_ratio")
    )
    .orderBy(F.desc("stockout_ratio"))
)

stockout_kpi.show(30, truncate=False)

### Ejercicio propuesto

1. Explica por qué el `stockout_ratio` puede ser más informativo que el número bruto de días.
2. ¿Qué medidas podría tomar una empresa ante productos con un ratio elevado?
3. Enlaza este análisis con la gestión de inventario.


## 16. Días con ventas y stock final nulo

Una situación especialmente interesante es aquella en la que hay ventas durante el día pero el stock final es cero o negativo.

Este patrón puede reflejar:
- fuerte rotación,
- agotamiento del inventario,
- o necesidad de revisar el aprovisionamiento.


In [ ]:
df_sales_stock_alert = retail_df.filter(
    (F.col("sales") > 0) & (F.col("stock") <= 0)
)

df_sales_stock_alert.select(
    "store_id", "product_id", "date", "sales", "revenue", "stock"
).show(30, truncate=False)

### Ejercicio propuesto

1. Interpreta el significado de una fila devuelta por este filtro.
2. ¿Crees que esta situación siempre es negativa? Justifica tu respuesta.
3. Calcula cuántos casos de este tipo hay por tienda.


## 17. Activos e inactivos mediante `left_semi` y `left_anti`

El dataset no dispone de `customer_id`, por lo que se adapta el patrón clásico de clientes activos/inactivos a pares `store_id` + `product_id`.

### Idea del análisis

- Un par tienda-producto se considera **activo** si aparece en los últimos 30 días del dataset.
- Se considera **inactivo** si no aparece en ese periodo.

Esto permite practicar dos joins muy relevantes en Spark:
- `left_semi`: devuelve las filas del lado izquierdo que sí tienen coincidencia.
- `left_anti`: devuelve las filas del lado izquierdo que no tienen coincidencia.


In [ ]:
max_date = retail_df.agg(F.max("date").alias("max_date")).collect()[0]["max_date"]

recent_df = retail_df.filter(F.col("date") >= F.date_sub(F.lit(max_date), 30))

store_product_pairs = retail_df.select("store_id", "product_id").distinct()
recent_pairs = recent_df.select("store_id", "product_id").distinct()

df_active_pairs = store_product_pairs.join(
    recent_pairs,
    ["store_id", "product_id"],
    "left_semi"
)

df_inactive_pairs = store_product_pairs.join(
    recent_pairs,
    ["store_id", "product_id"],
    "left_anti"
)

print("Pares tienda-producto activos:", df_active_pairs.count())
print("Pares tienda-producto inactivos:", df_inactive_pairs.count())

### Ejercicio propuesto

1. Explica la diferencia entre `left_semi`, `left_anti` e `inner join`.
2. ¿Qué aplicaciones tendría este patrón en un entorno retail?
3. Repite el análisis usando un umbral de 60 días.


## 18. Join por rango temporal con campañas simuladas

El dataset incluye información promocional a nivel de fila, pero para practicar un `range join` se construye una tabla auxiliar de campañas por periodos.

Una fila de ventas quedará asociada a una campaña si su fecha está comprendida entre:
- la fecha de inicio,
- y la fecha de fin del periodo.

Este tipo de join es útil en escenarios de:
- campañas,
- periodos de validez,
- reglas activas,
- ventanas temporales de negocio.


In [ ]:
promo_periods = spark.createDataFrame([
    ("campaign_1", "2017-01-01", "2017-06-30"),
    ("campaign_2", "2017-07-01", "2018-06-30"),
    ("campaign_3", "2018-07-01", "2019-12-31")
], ["campaign_id", "start_date", "end_date"])

promo_periods = (
    promo_periods
    .withColumn("start_date", F.to_date("start_date"))
    .withColumn("end_date", F.to_date("end_date"))
)

cond = (
    (retail_df["date"] >= promo_periods["start_date"]) &
    (retail_df["date"] < promo_periods["end_date"])
)

retail_campaign = retail_df.join(F.broadcast(promo_periods), cond, "left")

retail_campaign.select(
    "store_id", "product_id", "date", "revenue", "campaign_id"
).show(20, truncate=False)

### Ejercicio propuesto

1. Explica por qué este join no es un equi-join clásico.
2. ¿Qué problemas podrían aparecer si los periodos se solapan?
3. Añade una agregación que calcule la facturación total por `campaign_id`.


## 19. KPIs finales

En este bloque se construyen varios indicadores finales que resumen el comportamiento del dataset desde distintas perspectivas:

- por jerarquía principal de producto,
- por tamaño de tienda,
- por combinación de promociones.

Este tipo de salidas puede utilizarse como base para las conclusiones  o como tablas auxiliares para gráficos.


In [ ]:
kpi_h1 = (
    retail_df
    .groupBy("hierarchy1_id")
    .agg(
        F.sum("sales").alias("sales_qty"),
        F.sum("revenue").alias("revenue_total")
    )
    .orderBy(F.desc("revenue_total"))
)

kpi_store_size = (
    retail_df
    .groupBy("store_size")
    .agg(
        F.sum("sales").alias("sales_qty"),
        F.sum("revenue").alias("revenue_total")
    )
    .orderBy(F.desc("revenue_total"))
)

kpi_promos = (
    retail_df
    .groupBy("promo_type_1", "promo_type_2")
    .agg(
        F.sum("sales").alias("sales_qty"),
        F.sum("revenue").alias("revenue_total")
    )
    .orderBy(F.desc("revenue_total"))
)

print("KPIs por hierarchy1_id")
kpi_h1.show(20, truncate=False)

print("KPIs por tamaño de tienda")
kpi_store_size.show(20, truncate=False)

print("KPIs por promociones")
kpi_promos.show(20, truncate=False)

### Ejercicio propuesto

1. Resume qué dimensión parece más explicativa del comportamiento de ventas.
2. Selecciona tres KPIs y redacta una breve interpretación como si formara parte del TFG.
3. Propón dos gráficos para acompañar estos resultados.


## 20. Guardado opcional en Parquet

En entornos reales es habitual persistir resultados intermedios o finales en formatos columnares como Parquet.

Frente a CSV, Parquet ofrece ventajas como:
- compresión,
- mejor rendimiento de lectura,
- y preservación de tipos de datos.


In [ ]:
retail_df.write.mode("overwrite").parquet("out/retail_df")
df_pivot_promo1.write.mode("overwrite").parquet("out/df_pivot_promo1")
df_top_h1.write.mode("overwrite").parquet("out/df_top_h1")
stockout_kpi.write.mode("overwrite").parquet("out/stockout_kpi")

### Ejercicio propuesto

1. Explica por qué Parquet suele ser preferible a CSV en pipelines analíticos.
2. Investiga qué efecto tiene el particionado en Parquet sobre el rendimiento.
3. Propón una estrategia de particionado razonable para este dataset.


## 21. Cierre de la sesión de Spark
Al finalizar el análisis, es recomendable cerrar la sesión de Spark para liberar recursos.


In [ ]:
spark.stop()